# Collecting and Treating Data from a URL in Python

This notebook walks through the full pipeline:
1. Fetch raw data from a public URL
2. Parse and inspect it
3. Clean and transform it
4. Analyze it
5. Visualize it

## 1. Import Required Libraries

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import io

print("Libraries loaded successfully!")

Matplotlib is building the font cache; this may take a moment.


Libraries loaded successfully!


## 2. Fetch Data from a URL

We'll use a public CSV dataset of world population data hosted on GitHub.  
`requests.get()` sends an HTTP GET request. We check the status code — `200` means success.

In [2]:
url = "https://raw.githubusercontent.com/datasets/population/main/data/population.csv"

response = requests.get(url)

# Check the response
print(f"Status code : {response.status_code}")
print(f"Content type: {response.headers['Content-Type']}")
print(f"Size        : {len(response.content)} bytes")
print(f"\nFirst 300 characters of raw content:\n{response.text[:300]}")

Status code : 200
Content type: text/plain; charset=utf-8
Size        : 538674 bytes

First 300 characters of raw content:
Country Name,Country Code,Year,Value
Aruba,ABW,1960,54922
Aruba,ABW,1961,55578
Aruba,ABW,1962,56320
Aruba,ABW,1963,57002
Aruba,ABW,1964,57619
Aruba,ABW,1965,58190
Aruba,ABW,1966,58694
Aruba,ABW,1967,58990
Aruba,ABW,1968,59069
Aruba,ABW,1969,59052
Aruba,ABW,1970,58950
Aruba,ABW,1971,58781


## 3. Parse and Inspect the Data

We pass the raw text content directly into `pd.read_csv()` using `io.StringIO` — no need to save a file to disk.

In [3]:
# Parse the CSV content directly from memory (no file needed)
df = pd.read_csv(io.StringIO(response.text))

print("=== First 5 rows ===")
print(df.head())

print("\n=== DataFrame info ===")
df.info()

print("\n=== Basic statistics ===")
print(df.describe())

=== First 5 rows ===
  Country Name Country Code  Year    Value
0        Aruba          ABW  1960  54922.0
1        Aruba          ABW  1961  55578.0
2        Aruba          ABW  1962  56320.0
3        Aruba          ABW  1963  57002.0
4        Aruba          ABW  1964  57619.0

=== DataFrame info ===
<class 'pandas.DataFrame'>
RangeIndex: 16930 entries, 0 to 16929
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Name  16930 non-null  str    
 1   Country Code  16930 non-null  str    
 2   Year          16930 non-null  int64  
 3   Value         16930 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 529.2 KB

=== Basic statistics ===
               Year         Value
count  16930.000000  1.693000e+04
mean    1991.530124  2.166057e+08
std       18.472387  7.115080e+08
min     1960.000000  2.715000e+03
25%     1976.000000  1.009540e+06
50%     1992.000000  6.748606e+06
75%     2008.00000

## 4. Clean and Transform the Data

- Rename columns to be more Pythonic (lowercase, underscores)
- Drop rows with missing values
- Filter to keep only a subset of countries for focus

In [4]:
# Rename columns to snake_case
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
print("Renamed columns:", df.columns.tolist())

# Check for missing values
print(f"\nMissing values per column:\n{df.isnull().sum()}")

# Drop rows with any missing values
df = df.dropna()
print(f"\nRows after dropping NaN: {len(df)}")

# Filter: keep only a few countries for comparison
countries = ["United States", "China", "India", "Brazil", "Germany"]
df_filtered = df[df["country_name"].isin(countries)].copy()

# Convert year to integer (if not already)
df_filtered["year"] = df_filtered["year"].astype(int)

print(f"\nFiltered dataset shape: {df_filtered.shape}")
print(df_filtered.head(10))

Renamed columns: ['country_name', 'country_code', 'year', 'value']

Missing values per column:
country_name    0
country_code    0
year            0
value           0
dtype: int64

Rows after dropping NaN: 16930

Filtered dataset shape: (320, 4)
     country_name country_code  year       value
1856       Brazil          BRA  1960  72388126.0
1857       Brazil          BRA  1961  74605447.0
1858       Brazil          BRA  1962  76865323.0
1859       Brazil          BRA  1963  79164235.0
1860       Brazil          BRA  1964  81488595.0
1861       Brazil          BRA  1965  83817583.0
1862       Brazil          BRA  1966  86139359.0
1863       Brazil          BRA  1967  88446124.0
1864       Brazil          BRA  1968  90741240.0
1865       Brazil          BRA  1969  93045777.0


## 5. Analyze the Data

Use `groupby()` to find the latest population for each country, and compute year-over-year growth.

In [5]:
# Latest population per country
latest_year = df_filtered["year"].max()
latest = df_filtered[df_filtered["year"] == latest_year].copy()
latest = latest.sort_values("value", ascending=False)
latest["population_millions"] = (latest["value"] / 1_000_000).round(2)

print(f"=== Population in {latest_year} (millions) ===")
print(latest[["country_name", "population_millions"]].to_string(index=False))

# Year-over-year growth for the US
us = df_filtered[df_filtered["country_name"] == "United States"].sort_values("year")
us = us.copy()
us["yoy_growth_%"] = us["value"].pct_change() * 100

print("\n=== US Year-over-Year Population Growth (last 10 years) ===")
print(us[["year", "value", "yoy_growth_%"]].tail(10).to_string(index=False))

=== Population in 2023 (millions) ===
 country_name  population_millions
        India              1438.07
        China              1410.71
United States               334.91
       Brazil               211.14
      Germany                83.28

=== US Year-over-Year Population Growth (last 10 years) ===
 year       value  yoy_growth_%
 2014 318386329.0      0.736057
 2015 320738994.0      0.738934
 2016 323071755.0      0.727308
 2017 325122128.0      0.634649
 2018 326838199.0      0.527824
 2019 328329953.0      0.456420
 2020 331526933.0      0.973710
 2021 332048977.0      0.157467
 2022 333271411.0      0.368149
 2023 334914895.0      0.493137


## 6. Visualize the Data

Two charts:
- **Line chart** — Population over time per country
- **Bar chart** — Population comparison for the most recent year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Chart 1: Line chart — Population over time ---
for country in countries:
    subset = df_filtered[df_filtered["country_name"] == country]
    axes[0].plot(subset["year"], subset["value"] / 1_000_000, label=country, linewidth=2)

axes[0].set_title("Population Over Time", fontsize=14)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Population (millions)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- Chart 2: Bar chart — Latest year comparison ---
axes[1].bar(latest["country_name"], latest["population_millions"], color=["steelblue", "tomato", "seagreen", "orange", "mediumpurple"])
axes[1].set_title(f"Population Comparison ({latest_year})", fontsize=14)
axes[1].set_xlabel("Country")
axes[1].set_ylabel("Population (millions)")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

for i, (val, name) in enumerate(zip(latest["population_millions"], latest["country_name"])):
    axes[1].text(i, val + 10, f"{val:.0f}M", ha="center", fontsize=9)

plt.tight_layout()
plt.show()